In [ ]:
import os
import glob
import json
import time
import arcgis
from arcgis.gis import GIS
from arcgis.gis import Item

### Setup Environmental Variables

In [ ]:
arcgis.env.verbose=True

In [ ]:
# Some Setup information
profile = None
#'your_enterprise_profile'#'your_ortho_mapping_machine'#
#portalUrl = "https://sha-imgcl-d01.esri.com/portal"
#portalUN = "admin"
#portalPW = "adminadmin"
imageSuffix = ".jpg"
try:
    from utils import NOTEBOOK_TESTS_DIR
    imageFolderPath = os.path.join(NOTEBOOK_TESTS_DIR, "orthomapping", "BU")
except:
    imageFolderPath = r"./BU"

#arcgis.env.verbose =  False

In [ ]:
gis = GIS(url="https://ragsebtest01.esri.com/portal", username="andrew", password="esri.agp1",verify_cert=False)
gis

In [ ]:
from arcgis.raster import analytics
isSupport = analytics.is_supported(gis)

In [ ]:
gis.properties.helperServices

In [ ]:
if isSupport:
    print("Your enterprise gis supports orthomapping service")
else:
    print("Your enterprise gis does not support orthomapping service")
    exit()

### Check if Tool Creates

- This drives the whole testing documentation

In [ ]:
from arcgis._impl.tools import _OrthoMappingTools
omt = gis._tools.orthomapping
assert isinstance(omt, _OrthoMappingTools)

### Create Image Collection for Testing

In [ ]:
def currentTime():
    return time.strftime("%Y%m%d%H%M%S", time.localtime())
prjFolderName = "omProject" + currentTime()
gis.content.create_folder(folder=prjFolderName, owner=gis.users.me.username)
prjFolderName

In [ ]:
# Add Images and Create Image Collection
username = gis.users.me.username
imageList = glob.glob(os.path.join(imageFolderPath, '*.JPG'), recursive=True)
imageItemList = []
itemPropTemplate = {"type": "Image"}

for imageFullPath in imageList:
    imageName = imageFullPath[imageFullPath.rfind("\\")+1:]
    itemPropTemplate["title"] = imageName
    itemPropTemplate["tags"] = imageName
    itemPropTemplate["description"] = imageName

    imageItem = gis.content.add(item_properties=itemPropTemplate, data=imageFullPath,
                                owner=username, folder=prjFolderName)
    imageItemList.append(imageItem)

In [ ]:
swRasterTypeParams = {"gps": [['YUN_0040.JPG', 34.0069887, -117.09279029999999],
  ['YUN_0041.JPG', 34.0070131, -117.09311519972222],
  ['YUN_0042.JPG', 34.0070381, -117.09346329972222],
  ['YUN_0043.JPG', 34.00706339972222, -117.09381479999999],
  ['YUN_0044.JPG', 34.0070879, -117.09416449999999],
  ['YUN_0045.JPG', 34.007113099722226, -117.09450929972222],
  ['YUN_0046.JPG', 34.0071384, -117.09485779972222],
  ['YUN_0076.JPG', 34.00668639972222, -117.09463709972222],
  ['YUN_0077.JPG', 34.00666089972222, -117.09428809972222],
  ['YUN_0078.JPG', 34.00663709972222, -117.09395939999999],
  ['YUN_0079.JPG', 34.0066113, -117.09360669972222],
  ['YUN_0080.JPG', 34.00658549972222, -117.09325299999999],
  ['YUN_0081.JPG', 34.0065606, -117.0929043]],
"cameraProperties":{"maker":"Yuneec","model":"E90","focallength":8,"columns":5472,"rows":3648,"pixelsize":0.0024},
"isAltitudeFlightHeight":"false",
"averagezdem": {"url": "https://rais.dev.geocloud.com/arcgis/rest/services/Hosted/WorldSRTM90m/ImageServer"}}

In [ ]:
from arcgis.raster.analytics import create_image_collection
image_collection_name = "imgcollect" + currentTime()
image_collection = create_image_collection(image_collection=image_collection_name,
                                           input_rasters=imageItemList,
                                           raster_type_name="UAV/UAS",
                                           raster_type_params=swRasterTypeParams,
                                           out_sr=32632,
                                           gis=gis)

### Alter Processing States


In [ ]:
job_aps = omt.alter_processing_states(image_collection=image_collection, 
                                      new_states={"blockadjustment": "raw","dem": "Dense_Natual_Neighbor","seamlines":"VORONOI","colorcorrection":"SingleColor"}, 
                                      gis=gis, future=True)
assert job_aps
assert job_aps.result()

In [ ]:
job_aps.result()

### Compute Color Correction

In [ ]:
ccc_job = omt.compute_color_correction(image_collection,
                                    color_correction_method="DODGING",
                                    dodging_surface="SECOND_ORDER",
                                    context={"skipRows": 10, "skipCols": 10, "reCalculateStats": "OVERWRITE"},
                                    gis = gis,
                                    future=True)
assert ccc_job
assert ccc_job.result()

### Compute Control Points

In [ ]:
job_ccp = omt.compute_control_points(image_collection,
                                     reference_image="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
                                     image_location_accuracy='Low',
                                     gis=gis,
                                     future=True)
assert job_ccp
assert job_ccp.result()

### Compute Seamlines

In [ ]:
context={"minRegionSize":100,"blendType":"Both","blendWidth":None,
         "blendUnit":"Pixels","requestSizeType":"Pixels",
         "requestSize":1000,"minThinnessRatio":0.05,"maxSilverSize":20}
cs_job = omt.compute_seamlines(image_collection=image_collection,
                      seamlines_method="DISPARITY",
                      context=context,
                      gis=gis,
                      future=True)
assert cs_job
assert cs_job.result()

### Compute Sensor Model

In [ ]:
csm_job = omt.compute_sensor_model(image_collection, mode='QUICK', location_accuracy="Medium", future=True)
assert csm_job
assert csm_job.result()

### Edit Control Points

In [ ]:
inputControlPoints = [{"status":1,"type":2,"gcpid":"GCP9","tag":"GCP9","x":-117.0926538,
                      "y":34.00704253,"z":634.2175,"mapx":-13034694.596648848,
                      "mapy":4029747.7050537546,"spatialReference":{"wkid":4326},
                      "id":1,"pointId":1,"xyAccuracy":"0.008602325","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":12,"x":3458.3502468937245,"y":-4424.879721170419,
                                      "u":562.4306077298716,"v":-124.96945105501322},
                                     {"imageID":1,"x":2453.4233514297666,"y":-2001.2854377410533,
                                      "u":3056.8887978139765,"v":-1908.7056791900693},
                                     {"imageID":2,"x":1784.7197170631089,"y":-1114.535011913254,
                                      "u":3058.6565174539687,"v":-2960.846400950005},
                                     {"imageID":13,"x":4672.717414590265,"y":-2871.3511238888987,
                                      "u":617.2496070314755,"v":-1243.7106361762521}],"nlinks":4,"hasphoto":True},
                     {"status":1,"type":2,"gcpid":"GCP26","tag":"GCP26","x":-117.0934208,"y":34.00643003,"z":633.5438,
                      "mapx":-13034779.978698289,"mapy":4029665.454745273,"spatialReference":{"wkid":4326},
                      "id":5,"pointId":5,"xyAccuracy":"0.00781025","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":2,"x":5004.379068578275,"y":-1272.9293022678344,
                                      "u":563.4855594058413,"v":-867.580137100088}],
                      "nlinks":1,"hasphoto":True}]


In [ ]:
ecp_job = omt.edit_control_points(image_collection=image_collection,
                        input_control_points=inputControlPoints, 
                       gis=gis, future=True)
assert ecp_job
assert ecp_job.result()

### Generate DEM

In [ ]:
gen_dem_job = omt.generate_dem(image_collection=image_collection, output_name=None, 
                               cell_size=0.1314245599999937,
                               surface_type="DSM", 
                               matching_method="ETM",
                               context={"maxObjectSize":15,"minAngle":10,"maxAngle":70,"minOverlap":0.6,"maxGSDDif":2,"numImagePairs":8,"adjQualityThreshold":0.2,"method":"TRIANGULATION","smoothingMethod":"GAUSS5x5","applyToOrtho":False,"regenPointCloud":False}, 
                               gis=gis,
                              future=True)
assert gen_dem_job
assert isinstance(gen_dem_job.result(), Item)
assert gen_dem_job.result().delete()

### Generate Ortho Mosaic

In [ ]:
orthoMosaicName = "OMc" + currentTime() + "c"
othmos_job = omt.generate_orthomosaic(image_collection,
                         output_name=orthoMosaicName,
                         regen_seamlines=False, recompute_color_correction=False,
                         context=None,
                         gis=gis,
                         future=True)
assert othmos_job
res = othmos_job.result()
assert res
assert isinstance(res, Item)
assert res.delete()

### Generate Report

In [ ]:
report_link_job = omt.generate_report(image_collection=image_collection, gis=gis, future=True)
assert report_link_job
assert report_link_job.result()

### Get Processing States

In [ ]:
job_gps = omt.get_processing_states(image_collection = image_collection, future=True)
assert job_gps.result()

### Match Control Points

In [ ]:
inputControlPoints= [{"pointId":5,"x":-117.0934208,"y":34.00643003,"z":633.5438,
                      "xyAccuracy":"0.00781025","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":2,"x":5022.736523387883,"y":-1267.6047690326218}]}]
mcp_job = omt.match_control_points(image_collection=image_collection, 
                                   control_points=inputControlPoints,
                                   gis=gis, future=True)
assert mcp_job.result()

### Query Camera Info

In [ ]:
import pandas as pd
qci_job = omt.query_camera_info(query=None, future=True)
res = qci_job.result()
assert isinstance(res, pd.DataFrame)

### Query Control Points

In [ ]:
## Setup the process
ccp_job = omt.compute_control_points(image_collection=image_collection,
                       reference_image="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
                       image_location_accuracy='Low',
                       gis=gis, future=True)
ccp_job.result()
cp_job = omt.query_control_points(image_collection=image_collection.url,where="pointID>=0", future=True)
assert cp_job
assert cp_job.result()

### Reset Image collection

In [ ]:
reset_flag = omt.reset_image_collection(image_collection=image_collection, gis=gis, future=True)
assert reset_flag.result()

### Clean Up

In [ ]:
image_collection.delete()
for img in imageItemList:
    img.delete()

In [ ]:
gis.content.delete_folder(prjFolderName)